# Contexto de Execucao

Em muitas aplicacoes reais, o agente precisa de informacoes sobre o usuario ou o sistema sem que o usuario precise dize-las explicitamente. Por exemplo: um chatbot imobiliario pode precisar saber a cidade do cliente, seu orcamento maximo e o tipo de imovel de interesse. Essas informacoes ja existem no sistema (banco de dados, sessao do usuario) e nao faz sentido pedir ao usuario que repita.

O **contexto de execucao** (`context_schema`) resolve exatamente esse problema. Ele permite injetar informacoes externas na execucao do agente, acessiveis pelas tools via `runtime.context`. O contexto e **read-only**: as tools podem ler, mas nao podem modificar.

In [ ]:
from dotenv import load_dotenv

load_dotenv()

## Definindo o contexto

O contexto e definido como uma classe Pydantic (`BaseModel`). Cada campo representa uma informacao que o sistema conhece sobre o usuario. Vamos criar um contexto para um cliente de uma plataforma imobiliaria.

In [ ]:
from pydantic import BaseModel

class ContextoCliente(BaseModel):
    cidade: str = "Rio de Janeiro"
    orcamento_maximo: float = 500000.0
    tipo_imovel: str = "apartamento"

## Agente com contexto (sem tools)

Vamos primeiro testar o que acontece quando criamos um agente com `context_schema` mas sem nenhuma tool para acessar o contexto. O agente sabe que o contexto existe, mas nao tem como acessar os valores.

In [ ]:
from langchain.agents import create_agent

agente = create_agent(
    model="gpt-4.1-nano",
    context_schema=ContextoCliente
)

In [ ]:
from langchain.messages import HumanMessage

resposta = agente.invoke(
    {"messages": [HumanMessage(content="Qual e a minha cidade?")]},
    context=ContextoCliente()
)

print(resposta["messages"][-1].content)

Sem tools, o agente nao consegue acessar o contexto. Ele pode ate saber que existe um `ContextoCliente`, mas nao tem mecanismo para ler os valores. Precisamos criar tools que acessem `runtime.context`.

## Acessando o contexto via tools

Para que o agente acesse as informacoes do contexto, criamos tools que recebem `ToolRuntime` como parametro. Atraves de `runtime.context`, a tool le os campos do dataclass.

In [ ]:
from langchain.tools import tool, ToolRuntime

@tool
def obter_cidade(runtime: ToolRuntime) -> str:
    """Retorna a cidade do cliente."""
    return runtime.context.cidade

@tool
def obter_orcamento(runtime: ToolRuntime) -> str:
    """Retorna o orcamento maximo do cliente."""
    return f"R$ {runtime.context.orcamento_maximo:,.2f}"

@tool
def obter_tipo_imovel(runtime: ToolRuntime) -> str:
    """Retorna o tipo de imovel de interesse do cliente."""
    return runtime.context.tipo_imovel

Agora criamos o agente passando as tres tools e o `context_schema`. O agente tera acesso ao contexto por meio das ferramentas.

In [ ]:
agente = create_agent(
    model="gpt-4.1-nano",
    tools=[obter_cidade, obter_orcamento, obter_tipo_imovel],
    context_schema=ContextoCliente
)

In [ ]:
resposta = agente.invoke(
    {"messages": [HumanMessage(content="Qual e a minha cidade e meu orcamento?")]},
    context=ContextoCliente()
)

print(resposta["messages"][-1].content)

Agora o agente consegue acessar os dados do contexto. Ele chamou as tools `obter_cidade` e `obter_orcamento`, que leram os valores diretamente do `runtime.context`.

In [ ]:
from pprint import pprint

pprint(resposta["messages"])

## Sobrescrevendo valores do contexto

O contexto e definido **externamente**, por quem invoca o agente. Podemos passar valores diferentes a cada chamada, simulando usuarios distintos. O mesmo agente com as mesmas tools retorna resultados diferentes dependendo do contexto.

In [ ]:
outro_cliente = ContextoCliente(
    cidade="Florianopolis",
    orcamento_maximo=800000.0,
    tipo_imovel="casa"
)

resposta = agente.invoke(
    {"messages": [HumanMessage(content="Resuma meu perfil de busca.")]},
    context=outro_cliente
)

print(resposta["messages"][-1].content)

O mesmo agente, com as mesmas tools, retornou informacoes diferentes porque o contexto mudou. Esse padrao e muito util em aplicacoes multi-usuario, onde cada sessao tem seu proprio perfil.

O ponto principal e que o contexto e **read-only** e **definido externamente**. As tools podem ler os valores, mas nao podem altera-los durante a execucao. No proximo notebook, vamos ver o **estado do agente**, que e o complemento do contexto: dados que o proprio agente pode ler **e escrever**.